### Notebook 07 — Retention Agent


##### 1. Purpose

This notebook implements the Retention Agent for the multi-agent
customer support system.

The Retention Agent:

1. Reads its assigned task from the Coordinator execution plan.
2. Validates that required agent dependencies have completed.
3. Retrieves Prediction Agent and Vector Search Agent results
   from Shared State.
4. Builds a grounded retention context.
5. Calls an injected Retention Tool.
6. Validates and normalizes the tool response.
7. Creates a validated RetentionAgentResult.
8. Stores the result in Shared State.

The Retention Agent does not call the Prediction Agent or
Vector Search Agent directly. It consumes their previously
validated outputs from Shared State.


##### 2. Technologies Used

- Python
- Pydantic
- TypedDict shared state
- Callable tool contracts
- Dependency injection
- Agent dependency validation
- Structured agent communication
- Mock tools for unit testing


##### 3. Input

The Retention Agent receives:

- MultiAgentState
- Coordinator execution plan
- PredictionAgentResult from Shared State
- Optional VectorSearchAgentResult from Shared State
- Injected Retention Tool


##### 4. Output

The Retention Agent returns a validated RetentionAgentResult
containing:

- Customer ID
- Recommended retention action
- Reason for the recommendation
- Churn prediction label
- Prediction confidence
- Supporting customer notes
- Execution status and message


##### 5. Architecture


``` text

Coordinator Agent
        |
        ▼
Retention task and dependencies
        |
        ▼
Shared State
        |
        +-- PredictionAgentResult
        |
        +-- VectorSearchAgentResult
        |
        ▼
Retention Agent
        |
        +-- Validate dependencies
        +-- Build retention context
        +-- Call injected Retention Tool
        +-- Validate tool response
        |
        ▼
RetentionAgentResult
        |
        ▼
Shared State

```


##### 6. Load Shared Models and Helpers

In [0]:
%run ./01_shared_models_code_only

In [0]:
%run ./02_shared_state_and_helpers_code_only


##### 7. Imports

In [0]:
import re

from typing import Any, Callable, Dict, List, Optional


##### 8. Retention Tool Contract

In [0]:
RetentionToolFunction = Callable[
    [Dict[str, Any]],
    Dict[str, Any],
]
"""
Callable contract for the injected Retention Tool.

Input:
    A retention context containing validated customer,
    prediction, and supporting note information.

Output:
    A dictionary containing the retention recommendation
    and supporting reason.
"""

This contract means that any injected Retention Tool must:

- Accept:  Dict[str, Any]
- Return:  Dict[str, Any]

The tool implementation can later be replaced with:

- deterministic business rules,
- an LLM-backed tool,
- an external service,
- or a hybrid implementation.

The Retention Agent does not need to change as long as the new tool follows the same contract.


##### 9. Agent Constants

In [0]:
# Use the same values already defined in your shared models or agent notebooks.

RETENTION_AGENT_NAME = "retention_agent"

PREDICTION_AGENT_NAME = "prediction_agent"

VECTOR_SEARCH_AGENT_NAME = "vector_search_agent"

SUCCESS_STATUS = "success"

FAILED_STATUS = "failed"

SKIPPED_STATUS = "skipped"

RETENTION_ERROR_CODE = "RETENTION_FAILED"

In [0]:
# Allowed retention actions:

VALID_RETENTION_ACTIONS = {
    "offer_discount",
    "offer_support_package",
    "service_quality_review",
    "billing_review",
    "no_action",
}


##### 10. Find the Retention Agent Task

In [0]:
def find_retention_agent_task(
    coordinator_result: CoordinatorResult,
) -> Optional[AgentTask]:
    """
    Find the task assigned to the Retention Agent.
    """

    return get_agent_task(
        tasks=coordinator_result.execution_plan,
        agent_name=RETENTION_AGENT_NAME,
    )

##### 11. Read Dependency Results from Shared State

###### Prediction Agent result

In [0]:
def get_prediction_agent_result(
    state: MultiAgentState,
) -> PredictionAgentResult:
    """
    Retrieve and validate the Prediction Agent result
    from Shared State.
    """

    prediction_result = (
        state["agent_results"].get(
            PREDICTION_AGENT_NAME
        )
    )

    if prediction_result is None:
        raise ValueError(
            "Prediction Agent result was not found in Shared State."
        )

    if not isinstance(
        prediction_result,
        PredictionAgentResult,
    ):
        raise TypeError(
            "Prediction Agent result has an invalid type."
        )

    return prediction_result


###### Optional Vector Search Agent result

In [0]:
def get_vector_search_agent_result(
    state: MultiAgentState,
) -> Optional[VectorSearchAgentResult]:

    vector_search_result = (
        state["agent_results"].get(
            VECTOR_SEARCH_AGENT_NAME
        )
    )

    if vector_search_result is None:
        return None

    if not isinstance(
        vector_search_result,
        VectorSearchAgentResult,
    ):
        raise TypeError(
            "Vector Search Agent result has an invalid type."
        )

    return vector_search_result


##### 12. Build Supporting Notes

In [0]:
def build_supporting_notes(
    vector_search_result: Optional[
        VectorSearchAgentResult
    ],
) -> List[str]:
    """
    Extract non-empty customer notes from the
    Vector Search Agent result.
    """

    if vector_search_result is None:
        return []

    supporting_notes = []

    for search_item in vector_search_result.results:
        note = str(
            search_item.note
        ).strip()

        if note:
            supporting_notes.append(note)

    return supporting_notes

##### 13. Build the Retention Context from Shared State

In [0]:
def build_retention_context(
    state: MultiAgentState,
) -> Dict[str, Any]:
    """
    Build the information passed to the Retention Tool
    using validated results stored in Shared State.
    """

    prediction_result = get_prediction_agent_result(
        state=state
    )

    vector_search_result = (
        get_vector_search_agent_result(
            state=state
        )
    )

    if prediction_result.status != SUCCESS_STATUS:
        raise ValueError(
            "Retention Agent cannot continue because the "
            "Prediction Agent did not complete successfully."
        )

    if (
        vector_search_result is not None
        and vector_search_result.status != SUCCESS_STATUS
    ):
        raise ValueError(
            "Vector Search Agent result exists but did not "
            "complete successfully."
        )

    customer_id = prediction_result.customer_id

    if not customer_id:
        raise ValueError(
            "Prediction Agent result is missing customer_id."
        )

    prediction_label = (
        prediction_result.predicted_category
    )

    prediction_confidence = (
        prediction_result.confidence
    )

    if prediction_label is None:
        raise ValueError(
            "Prediction Agent result is missing the "
            "prediction label."
        )

    if prediction_confidence is None:
        raise ValueError(
            "Prediction Agent result is missing confidence."
        )

    supporting_notes = build_supporting_notes(
        vector_search_result=vector_search_result
    )

    return {
        "customer_id": customer_id,
        "prediction_label": prediction_label,
        "prediction_confidence": prediction_confidence,
        "supporting_notes": supporting_notes,
    }

##### 14. Validate the Retention Tool Response

In [0]:
def validate_retention_tool_response(
    tool_response: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Validate and normalize the raw dictionary returned
    by the Retention Tool.
    """

    if not isinstance(tool_response, dict):
        raise TypeError(
            "Retention Tool must return a dictionary."
        )

    status = tool_response.get(
        "status"
    )

    if status != SUCCESS_STATUS:
        error_message = tool_response.get(
            "message",
            "Retention Tool execution failed.",
        )

        raise RuntimeError(
            str(error_message)
        )

    recommended_action = tool_response.get(
        "recommended_action"
    )

    if recommended_action is None:
        raise ValueError(
            "Retention Tool response is missing "
            "'recommended_action'."
        )

    recommended_action = str(
        recommended_action
    ).strip()

    if recommended_action not in VALID_RETENTION_ACTIONS:
        raise ValueError(
            "Retention Tool returned an unsupported "
            f"retention action: {recommended_action!r}."
        )

    action_reason = tool_response.get(
        "action_reason",
        tool_response.get("reason"),
    )

    if action_reason is None:
        raise ValueError(
            "Retention Tool response is missing "
            "'action_reason'."
        )

    action_reason = str(
        action_reason
    ).strip()

    if not action_reason:
        raise ValueError(
            "Retention Tool returned an empty "
            "action reason."
        )

    return {
        "recommended_action": recommended_action,
        "action_reason": action_reason,
        "raw_tool_response": tool_response,
    }

##### 15. Execute the Retention Agent

In [0]:
def execute_retention_agent(
    state: MultiAgentState,
    retention_tool: RetentionToolFunction,
) -> Optional[RetentionAgentResult]:
    """
    Execute the task assigned to the Retention Agent.

    Returns None when no Retention Agent task was assigned.
    """

    coordinator_result = state.get(
        "coordinator_result"
    )

    if coordinator_result is None:
        raise ValueError(
            "Retention Agent cannot run because "
            "coordinator_result is missing."
        )

    task = find_retention_agent_task(
        coordinator_result=coordinator_result
    )

    if task is None:
        return None

    validate_task_dependencies(
        state=state,
        task=task,
    )

    retention_context = build_retention_context(
        state=state,
    )

    raw_tool_response = retention_tool(
        retention_context
    )

    validated_response = (
        validate_retention_tool_response(
            tool_response=raw_tool_response
        )
    )

    recommended_action = validated_response[
        "recommended_action"
    ]

    action_reason = validated_response[
        "action_reason"
    ]

    result = RetentionAgentResult(
        agent_name=RETENTION_AGENT_NAME,
        task_id=task.task_id,
        status=SUCCESS_STATUS,
        message=(
            "Retention Agent completed successfully "
            f"and recommended '{recommended_action}'."
        ),
        customer_id=retention_context[
            "customer_id"
        ],
        recommended_action=recommended_action,
        action_reason=action_reason,
        prediction_label=retention_context[
            "prediction_label"
        ],
        prediction_confidence=retention_context[
            "prediction_confidence"
        ],
        supporting_notes=retention_context[
            "supporting_notes"
        ],
    )

    return result

##### 16. Run the Retention Agent

In [0]:
def run_retention_agent(
    state: MultiAgentState,
    retention_tool: RetentionToolFunction,
) -> MultiAgentState:
    """
    Run the Retention Agent and update Shared State.
    """

    try:
        result = execute_retention_agent(
            state=state,
            retention_tool=retention_tool,
        )

        if result is None:
            
            add_execution_history(
                state=state,
                agent_name=RETENTION_AGENT_NAME,
                status="skipped",
                message=(
                    "Retention Agent skipped because no "
                    "retention task was assigned."
                ),
            )

            return state

        store_agent_result(
            state=state,
            agent_result=result,
        )

        add_execution_history(
            state=state,
            agent_name=RETENTION_AGENT_NAME,
            status="success",
            message=(
                "Retention Agent completed successfully."
            ),
        )

    except Exception as exc:
        error_message = (
            f"Retention Agent failed: {exc}"
        )

        add_error(
            state=state,
            agent_name=RETENTION_AGENT_NAME,
            error_code=RETENTION_ERROR_CODE,
            error_message=error_message,
        )

        add_execution_history(
            state=state,
            agent_name=RETENTION_AGENT_NAME,
            status="failed",
            message=error_message,
        )

    return state

##### 17. Mock Retention Tools

###### Successful rule-based tool

In [0]:
def mock_retention_tool(
    retention_context: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Mock rule-based Retention Tool used for learning
    and unit testing.

    Input:
        retention_context

    Output:
        Dictionary containing:
            - status
            - recommended_action
            - action_reason
    """

    prediction_label = retention_context[
        "prediction_label"
    ]

    prediction_confidence = retention_context[
        "prediction_confidence"
    ]

    supporting_notes = retention_context.get(
        "supporting_notes",
        [],
    )

    combined_notes = " ".join(
        supporting_notes
    ).lower()

    # --------------------------------------------------
    # Validate prediction label
    # --------------------------------------------------

    if prediction_label not in {
        "Churn",
        "No Churn",
    }:
        return {
            "status": "error",
            "message": (
                f"Unsupported prediction label: "
                f"{prediction_label!r}"
            ),
        }

    # --------------------------------------------------
    # Customer is not predicted to churn
    # --------------------------------------------------

    if prediction_label == "No Churn":
        return {
            "status": SUCCESS_STATUS,
            "recommended_action": "no_action",
            "action_reason": (
                "The customer is not currently "
                "predicted to churn."
            ),
        }

    # --------------------------------------------------
    # From this point onward, the customer is
    # guaranteed to be predicted to churn.
    # --------------------------------------------------

    if any(
        keyword in combined_notes
        for keyword in (
            "billing",
            "charge",
            "price",
            "cost",
        )
    ):
        return {
            "status": SUCCESS_STATUS,
            "recommended_action": "billing_review",
            "action_reason": (
                "The customer is predicted to churn "
                "and the supporting notes indicate "
                "billing or pricing concerns."
            ),
        }

    if any(
        keyword in combined_notes
        for keyword in (
            "slow",
            "buffer",
            "internet",
            "connection",
        )
    ):
        return {
            "status": SUCCESS_STATUS,
            "recommended_action": (
                "service_quality_review"
            ),
            "action_reason": (
                "The customer is predicted to churn "
                "and the supporting notes indicate "
                "service quality concerns."
            ),
        }

    if prediction_confidence >= 0.80:
        return {
            "status": SUCCESS_STATUS,
            "recommended_action": "offer_discount",
            "action_reason": (
                "The customer has a high predicted "
                "churn risk and no specific complaint "
                "was identified."
            ),
        }

    return {
        "status": SUCCESS_STATUS,
        "recommended_action": (
            "offer_support_package"
        ),
        "action_reason": (
            "The customer has moderate predicted "
            "churn risk and may benefit from "
            "additional support."
        ),
    }

###### Tool failure

In [0]:
def mock_retention_tool_failure(
    retention_context: Dict[str, Any],
) -> Dict[str, Any]:
    return {
        "status": FAILED_STATUS,
        "message": (
            "Mock Retention Tool execution failed."
        ),
    }

###### Invalid action

In [0]:
def mock_retention_tool_invalid_action(
    retention_context: Dict[str, Any],
) -> Dict[str, Any]:
    return {
        "status": SUCCESS_STATUS,
        "recommended_action": "free_phone",
        "action_reason": (
            "This action is not supported."
        ),
    }

###### Missing required field

In [0]:
def mock_retention_tool_missing_reason(
    retention_context: Dict[str, Any],
) -> Dict[str, Any]:
    return {
        "status": SUCCESS_STATUS,
        "recommended_action": "offer_discount",
    }

##### 18. Test Helpers

In [0]:
def create_mock_prediction_result(
    customer_id: str = "1001",
    prediction_label: str = "Churn",
    confidence: float = 0.91,
) -> PredictionAgentResult:
    """
    Create a successful Prediction Agent result.
    """

    prediction_data = {
        "agent_name": PREDICTION_AGENT_NAME,
        "status": SUCCESS_STATUS,
        "message": (
            "Prediction Agent completed successfully."
        ),
        "predicted_category": prediction_label,
        "confidence": confidence,
        "model_name": "mock_churn_model",
        "raw_prediction": (
            prediction_label == "Churn"
        ),
    }

    if (
        "customer_id"
        in PredictionAgentResult.model_fields
    ):
        prediction_data["customer_id"] = customer_id

    return PredictionAgentResult(
        **prediction_data
    )

In [0]:
def create_mock_vector_search_result(
    customer_id: str = "1001",
) -> VectorSearchAgentResult:
    """
    Create a successful Vector Search Agent result.
    """

    return VectorSearchAgentResult(
        agent_name=VECTOR_SEARCH_AGENT_NAME,
        task_id="vector-task-1",
        status=SUCCESS_STATUS,
        message=(
            "Vector Search Agent retrieved two notes."
        ),
        query=(
            "Find customer concerns related to churn."
        ),
        results=[
            VectorSearchItem(
                customer_id=customer_id,
                note=(
                    "Customer complained about high "
                    "monthly charges."
                ),
                similarity_score=0.94,
            ),
            VectorSearchItem(
                customer_id=customer_id,
                note=(
                    "Customer asked about cancelling "
                    "the service."
                ),
                similarity_score=0.88,
            ),
        ],
    )

##### 19. Coordinator Result for Tests

In [0]:
def create_retention_coordinator_result(
    include_retention_task: bool = True,
    depends_on: Optional[List[AgentName]] = None,
) -> CoordinatorResult:
    """
    Create a mock CoordinatorResult for Retention Agent tests.
    """

    execution_plan = []

    if include_retention_task:
        execution_plan.append(
            AgentTask(
                task_id="retention-task-1",
                agent_name=RETENTION_AGENT_NAME,
                task_description=(
                    "Recommend a retention action for "
                    "customer 1001."
                ),
                depends_on=(
                    depends_on
                    if depends_on is not None
                    else [
                        PREDICTION_AGENT_NAME,
                    ]
                ),
            )
        )

    return CoordinatorResult(
    agent_name="coordinator_agent",
    status=SUCCESS_STATUS,
    message="Coordinator execution plan created.",
    request_type="retention",
    reasoning=(
        "Customer retention recommendation is required."
    ),
    execution_plan=execution_plan,
)

##### 20. Tests

###### Test 1 — Successful recommendation

In [0]:
def test_successful_retention_recommendation():
    state = create_initial_state(
        user_request=(
            "Recommend a retention action for "
            "customer 1001."
        )
    )

    state["coordinator_result"] = (
        create_retention_coordinator_result()
    )

    state["agent_results"][
        PREDICTION_AGENT_NAME
    ] = create_mock_prediction_result()

    state["agent_results"][
        VECTOR_SEARCH_AGENT_NAME
    ] = create_mock_vector_search_result()

    updated_state = run_retention_agent(
        state=state,
        retention_tool=mock_retention_tool,
    )

    assert (
        RETENTION_AGENT_NAME
        in updated_state["agent_results"]
    )

    result = updated_state["agent_results"][
        RETENTION_AGENT_NAME
    ]

    assert isinstance(
        result,
        RetentionAgentResult,
    )

    assert result.status == SUCCESS_STATUS

    assert result.customer_id == "1001"

    assert result.recommended_action == (
        "billing_review"
    )

    assert result.prediction_label == "Churn"

    assert result.prediction_confidence == 0.91

    assert len(result.supporting_notes) == 2

    assert len(updated_state["errors"]) == 0

    print(
        "TEST 1 PASSED: "
        "Successful retention recommendation"
    )

###### Test 2 — No-action recommendation

In [0]:
def test_no_action_recommendation():
    state = create_initial_state(
        user_request=(
            "Recommend a retention action for "
            "customer 1001."
        )
    )

    state["coordinator_result"] = (
        create_retention_coordinator_result(
            depends_on=[
                PREDICTION_AGENT_NAME
            ]
        )
    )

    state["agent_results"][
        PREDICTION_AGENT_NAME
    ] = create_mock_prediction_result(
        prediction_label="No Churn",
        confidence=0.84,
    )

    updated_state = run_retention_agent(
        state=state,
        retention_tool=mock_retention_tool,
    )

    result = updated_state["agent_results"][
        RETENTION_AGENT_NAME
    ]

    assert result.status == SUCCESS_STATUS

    assert result.recommended_action == "no_action"

    assert result.prediction_label == "No Churn"

    assert result.supporting_notes == []

    assert len(updated_state["errors"]) == 0

    print(
        "TEST 2 PASSED: "
        "Successful no-action recommendation"
    )

###### Test 3 — Skip when no task is assigned

In [0]:
def test_retention_agent_skip():
    state = create_initial_state(
        user_request=(
            "How many customers churned?"
        )
    )

    state["coordinator_result"] = (
        create_retention_coordinator_result(
            include_retention_task=False
        )
    )

    updated_state = run_retention_agent(
        state=state,
        retention_tool=mock_retention_tool,
    )

    assert (
        RETENTION_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(updated_state["errors"]) == 0

    print(
        "TEST 3 PASSED: "
        "Retention Agent skipped successfully"
    )

###### Test 4 — Missing Prediction Agent dependency

In [0]:
def test_missing_prediction_dependency():
    """
    Verify that the Retention Agent records an error
    when the Prediction Agent result is missing.
    """

    state = create_initial_state(
        user_request=(
            "Recommend a retention action for "
            "customer 1001."
        )
    )

    state["coordinator_result"] = (
        create_retention_coordinator_result()
    )

    state["agent_results"][
        VECTOR_SEARCH_AGENT_NAME
    ] = create_mock_vector_search_result()

    updated_state = run_retention_agent(
        state=state,
        retention_tool=mock_retention_tool,
    )

    assert (
        RETENTION_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(updated_state["errors"]) == 1

    error = updated_state["errors"][0]

    assert (
        error.agent_name
        == RETENTION_AGENT_NAME
    )

    assert (
        "Prediction Agent"
        in error.error_message
        or PREDICTION_AGENT_NAME
        in error.error_message
    )

    assert len(
        updated_state["execution_history"]
    ) == 1

    history_entry = (
        updated_state["execution_history"][0]
    )

    assert (
        history_entry.agent_name
        == RETENTION_AGENT_NAME
    )

    assert history_entry.status == "failed"

    assert (
        RETENTION_AGENT_NAME
        in history_entry.agent_name
    )

    print(
        "TEST 4 PASSED: "
        "Missing Prediction Agent dependency handled"
    )

###### Test 5 — Retention Tool failure

In [0]:
def test_retention_tool_failure():
    """
    Verify that a Retention Tool failure is recorded
    correctly in Shared State.
    """

    state = create_initial_state(
        user_request=(
            "Recommend a retention action for "
            "customer 1001."
        )
    )

    state["coordinator_result"] = (
        create_retention_coordinator_result()
    )

    state["agent_results"][
        PREDICTION_AGENT_NAME
    ] = create_mock_prediction_result()

    state["agent_results"][
        VECTOR_SEARCH_AGENT_NAME
    ] = create_mock_vector_search_result()

    updated_state = run_retention_agent(
        state=state,
        retention_tool=mock_retention_tool_failure,
    )

    assert (
        RETENTION_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(updated_state["errors"]) == 1

    error = updated_state["errors"][0]

    assert (
        error.agent_name 
        == RETENTION_AGENT_NAME
    )

    assert (
        "Retention Agent failed"
        in error.error_message
    )

    assert len(
        updated_state["execution_history"]
    ) == 1

    history_entry = (
        updated_state["execution_history"][0]
    )

    assert (
        history_entry.agent_name
        == RETENTION_AGENT_NAME
    )

    assert history_entry.status == "failed"

    assert (
        "Retention Agent failed"
        in history_entry.message
    )

    print(
        "TEST 5 PASSED: "
        "Retention Tool failure handled"
    )

###### Test 6 — Invalid retention action

In [0]:
def test_invalid_retention_action():
    state = create_initial_state(
        user_request=(
            "Recommend a retention action for "
            "customer 1001."
        )
    )

    state["coordinator_result"] = (
        create_retention_coordinator_result()
    )

    state["agent_results"][
        PREDICTION_AGENT_NAME
    ] = create_mock_prediction_result()

    state["agent_results"][
        VECTOR_SEARCH_AGENT_NAME
    ] = create_mock_vector_search_result()

    updated_state = run_retention_agent(
        state=state,
        retention_tool=(
            mock_retention_tool_invalid_action
        ),
    )

    assert (
        RETENTION_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(updated_state["errors"]) == 1

    error = updated_state["errors"][0]

    assert (
        error.agent_name
        == RETENTION_AGENT_NAME
    )

    assert (
        "Retention Agent failed"
        in error.error_message
    )

    assert len(
        updated_state["execution_history"]
    ) == 1

    history_entry = (
        updated_state["execution_history"][0]
    )

    assert (
        history_entry.agent_name
        == RETENTION_AGENT_NAME
    )

    assert history_entry.status == "failed"

    assert (
        "Retention Agent failed"
        in history_entry.message
    )


    print(
        "TEST 6 PASSED: "
        "Invalid retention action handled"
    )

###### Test 7 — Missing action reason

In [0]:
def test_missing_action_reason():
    """
    Verify that a missing action_reason returned by the
    Retention Tool is handled correctly.
    """

    state = create_initial_state(
        user_request=(
            "Recommend a retention action for "
            "customer 1001."
        )
    )

    state["coordinator_result"] = (
        create_retention_coordinator_result()
    )

    state["agent_results"][
        PREDICTION_AGENT_NAME
    ] = create_mock_prediction_result()

    state["agent_results"][
        VECTOR_SEARCH_AGENT_NAME
    ] = create_mock_vector_search_result()

    updated_state = run_retention_agent(
        state=state,
        retention_tool=(
            mock_retention_tool_missing_reason
        ),
    )

    assert (
        RETENTION_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(updated_state["errors"]) == 1

    error = updated_state["errors"][0]

    assert (
        error.agent_name
        == RETENTION_AGENT_NAME
    )

    assert (
        "action_reason"
        in error.error_message
    )

    assert len(
        updated_state["execution_history"]
    ) == 1

    history_entry = (
        updated_state["execution_history"][0]
    )

    assert (
        history_entry.agent_name
        == RETENTION_AGENT_NAME
    )

    assert history_entry.status == "failed"

    assert (
        "action_reason"
        in history_entry.message
    )

    print(
        "TEST 7 PASSED: "
        "Missing action reason handled"
    )

###### 21. Run All Tests

In [0]:
def test_retention_agent():
    print("=" * 80)
    print("RETENTION AGENT TESTS")
    print("=" * 80)

    test_successful_retention_recommendation()

    test_no_action_recommendation()

    test_retention_agent_skip()

    test_missing_prediction_dependency()

    test_retention_tool_failure()

    test_invalid_retention_action()

    test_missing_action_reason()

    print("=" * 80)
    print("ALL RETENTION AGENT TESTS PASSED")
    print("=" * 80)

In [0]:
test_retention_agent()

##### 22. Inspect the Collaboration Through Shared State

In [0]:
#it shows exactly what the Retention Agent consumes.

demo_state = create_initial_state(
    user_request=(
        "Recommend a retention action for customer 1001."
    )
)

demo_state["coordinator_result"] = (
    create_retention_coordinator_result()
)

demo_state["agent_results"][
    PREDICTION_AGENT_NAME
] = create_mock_prediction_result()

demo_state["agent_results"][
    VECTOR_SEARCH_AGENT_NAME
] = create_mock_vector_search_result()

##### 23. Key Learnings

1. The Retention Agent does not directly call other agents.

2. It consumes validated PredictionAgentResult and VectorSearchAgentResult objects from Shared State.

3. Coordinator task dependencies determine which agent results must exist before the Retention Agent can execute.

4. The Retention Agent builds a focused retention context rather than passing the entire Shared State to the Retention Tool.

5. The Retention Tool is injected through a Callable contract.

6. Mock and rule-based tools support inexpensive, deterministic learning and testing.

7. The Retention Tool can later be replaced with an LLM-backed implementation without rewriting the Retention Agent.

8. Raw tool responses are manually validated and normalized.

9. RetentionAgentResult provides the trusted Pydantic contract stored in Shared State for downstream agents.

10. This notebook demonstrates real collaboration between specialist agents through Shared State.

##### 24. Conclusion

- The Retention Agent was successfully implemented as a dependent specialist agent.

- Unlike the SQL, Prediction, and Vector Search Agents, the Retention Agent consumes outputs produced by other agents.

- It retrieves validated PredictionAgentResult and VectorSearchAgentResult objects from Shared State, builds a grounded retention context, calls an injected Retention Tool,
validates the returned recommendation, and stores a validated RetentionAgentResult.

- This demonstrates how specialist agents collaborate without calling or tightly coupling themselves to one another.

##### 25. Next Notebook

Notebook 08 will implement the Final Response Agent.

The Final Response Agent will:

- Read validated results from Shared State.
- Determine which agent results are relevant.
- Build a grounded response context.
- Generate a clear user-facing answer.
- Avoid introducing information not supported by agent results.
- Return a validated FinalResponseAgentResult.

In [0]:
state = create_initial_state(
    user_request=(
        "Recommend a retention action for "
        "customer 1001."
    )
)

state["coordinator_result"] = (
    create_retention_coordinator_result()
)

state["agent_results"][
    VECTOR_SEARCH_AGENT_NAME
] = create_mock_vector_search_result()

updated_state = run_retention_agent(
    state=state,
    retention_tool=mock_retention_tool,
)

assert (
    RETENTION_AGENT_NAME
    not in updated_state["agent_results"]
)

assert len(updated_state["errors"]) == 1

error = updated_state["errors"][0]

assert (
    error.agent_name
    == RETENTION_AGENT_NAME
)

assert (
    "Prediction Agent"
    in error.error_message
    or PREDICTION_AGENT_NAME
    in error.error_message
)

assert len(
    updated_state["execution_history"]
) == 1

history_entry = (
    updated_state["execution_history"][0]
)

assert (
    history_entry.agent_name
    == RETENTION_AGENT_NAME
)

assert history_entry.status == "failed"

assert (
    RETENTION_AGENT_NAME
    in history_entry.agent_name
)

print(
    "TEST 4 PASSED: "
    "Missing Prediction Agent dependency handled"
)